# Metodologia 2: segmentação diretamente em 3D

Exemplo executável da metodologia de segmentação diretamente em 3D: extrair um esqueleto volumétrico, analisar a continuidade nas junções em 3D e expandir os marcadores por watershed. Diferentemente do método 2D, não existe propagação de IDs entre planos.

O roteiro usa as funções existentes, sem copiar seus algoritmos. Percorremos as etapas explicitamente e verificamos a equivalência com `Direct3DFiberPipeline`. Comprimento e orientação são medidos depois da segmentação com o módulo compartilhado.

**Execução:** rode as células em ordem no ambiente científico do projeto. Usamos um cubo central de 200 × 200 × 200 voxels de `data/PPGF40.tif`, sem salvar resultados automaticamente. O arquivo precisa estar disponível localmente. Os antigos resultados exploratórios não são reutilizados.

## 1. Preparar o ambiente e as convenções

Use um kernel com NumPy, SciPy, scikit-image, skan, pandas, Matplotlib, tifffile e Jupyter. Abra o notebook na raiz do repositório ou em `notebooks/`.

Arrays usam **(Z, Y, X)**; o espaçamento das medidas segue **(dz, dy, dx)**. As figuras mostram cortes XY em Z constante. Não há modelo treinado: a classe apenas organiza o processamento de imagem e a análise do esqueleto.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display
from scipy.ndimage import binary_opening
from skimage.color import label2rgb
from skimage.filters import threshold_otsu
from skimage.measure import label
from skimage.morphology import skeletonize

root = Path.cwd()
if not (root / "fiber_analysis").is_dir():
    root = root.parent
if not (root / "fiber_analysis").is_dir():
    raise FileNotFoundError("Abra o notebook na raiz do repositório ou em notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from fiber_analysis.config import Direct3DConfig
from fiber_analysis.metrics import measure_fibers
from fiber_analysis.pipelines import Direct3DFiberPipeline
from src.utils import analysis
from src.utils.preprocessing import auto_crop_volume, select_cubic_section
from src.utils.segmentation import label_fibers_watershed

## 2. Carregar e selecionar a entrada

Mapeamos `data/PPGF40.tif` em memória, em modo somente leitura, e selecionamos um cubo central de 200 voxels por aresta. Não há entrada sintética ou fallback para arquivo ausente.

Preservamos a normalização histórica: dividir pelo **máximo do TIFF inteiro**, multiplicar por 255 e converter para `uint8`. A busca do máximo percorre os dados, mas a conversão é feita somente no cubo selecionado, sem criar um volume completo em ponto flutuante. Isso produz os mesmos valores que normalizar o TIFF inteiro antes do recorte.

A seleção do cubo pertence ao experimento, não à classe. Não aplicamos a ROI específica do runner 2D.

In [ ]:
arquivo_tiff = root / "data" / "PPGF40.tif"
aresta_cubo = 200

volume_completo = tifffile.memmap(arquivo_tiff, mode="r")
intensidade_maxima = volume_completo.max()
cubo = select_cubic_section(volume_completo, pos="ct", n_pixels=aresta_cubo)
volume_selecionado = (cubo / intensidade_maxima * 255).astype(np.uint8)
del volume_completo, cubo

print(f"Entrada: {arquivo_tiff}")
print(f"Volume selecionado (Z, Y, X): {volume_selecionado.shape}")
print(f"Tipo: {volume_selecionado.dtype}")

## 3. Definir a configuração da análise

Explicitamos os parâmetros do exemplo de experimento do runner 3D: ramos terminais com mínimo 10, isolados com mínimo 15, janela local de 12 voxels, ângulo de 75° e retração de 4 voxels. Estes **não são todos os padrões** da classe: `Direct3DConfig()` usa mínimos 5/10, janela 10 e ângulo 45°.

Apesar do nome histórico `max_angle`, o valor é encaminhado como **`min_angle`** para a poda: um par deve formar um ângulo de pelo menos esse valor. Não é o desvio em relação a 180° usado pelo método 2D. O texto metodológico cita 45°; este exemplo deixa explícita a configuração de 75° do experimento, sem alterar nenhum critério do algoritmo.

In [ ]:
config = Direct3DConfig(
    min_branch_length=10,
    remove_isolated_branches=True,
    min_isolated_length=15,
    local_search=True,
    local_range=12,
    max_angle=75,
    retraction_dist=4,
)
print(config)

## 4. Remover margens, binarizar e aplicar abertura 3D

A sequência é a mesma de `Direct3DFiberPipeline`: `auto_crop_volume` remove margens cujas intensidades são zero; Otsu é calculado **no volume já recortado**; a comparação estrita `> limiar` produz a máscara; uma abertura morfológica 3D reduz pequenas protrusões.

O recorte automático não identifica o fundo por Otsu: ruído ou fundo positivo podem impedir a remoção de margens. Os próximos arrays usam o referencial do volume recortado, que pode diferir da entrada. Não aplicamos mediana, fechamento ou outro filtro adicional.

In [ ]:
volume_recortado = auto_crop_volume(volume_selecionado)
limiar = threshold_otsu(volume_recortado)
mascara_otsu = volume_recortado > limiar
mascara = binary_opening(mascara_otsu)
indice_z = volume_recortado.shape[0] // 2
print(f"Limiar de Otsu: {limiar}; voxels de foreground: {int(mascara.sum())}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].hist(volume_recortado.ravel(), bins=64, color="gray")
axes[0].axvline(limiar, color="red", label=f"Otsu = {limiar}")
axes[0].set(xlabel="Intensidade", ylabel="Voxels", title="Histograma após recorte")
axes[0].legend()
for ax, imagem, titulo in zip(
    axes[1:],
    (volume_recortado[indice_z], mascara[indice_z]),
    ("Tons de cinza", "Após abertura 3D"),
):
    ax.imshow(imagem, cmap="gray")
    ax.set(title=f"{titulo}: Z={indice_z}", xlabel="X", ylabel="Y")
fig.tight_layout()
plt.show()

## 5. Extrair o esqueleto volumétrico e localizar junções

`skeletonize` opera no volume 3D, reduzindo as estruturas à sua representação unifilar. O grafo do skan fornece os nós e caminhos usados na análise: EP é uma extremidade e JP é uma junção.

`analysis.junction_mask_3d` é apenas uma visualização do grafo: 0 representa fundo, 1 esqueleto e 2 nós com grau pelo menos 3. Não altera a máscara ou o esqueleto. O corte XY mostra apenas os voxels presentes naquele plano, não uma projeção do volume.

In [ ]:
esqueleto_inicial = skeletonize(mascara)
diagnostico_juncoes = analysis.junction_mask_3d(esqueleto_inicial)
print(f"Voxels do esqueleto: {int(esqueleto_inicial.sum())}")
print(f"Nós de junção: {int(np.count_nonzero(diagnostico_juncoes == 2))}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(esqueleto_inicial[indice_z], cmap="gray")
axes[0].set_title("Esqueleto inicial")
axes[1].imshow(
    label2rgb(diagnostico_juncoes[indice_z], colors=["white", "red"], bg_label=0)
)
axes[1].set_title("Esqueleto branco; junções vermelhas")
for ax in axes:
    ax.set(xlabel="X", ylabel="Y")
fig.tight_layout()
plt.show()

## 6. Filtrar ramos curtos antes da análise angular

`analysis.filter_skeleton` distingue os tipos de ramo pelo grafo. Remove JP–EP com menos de `min_branch_length` voxels, preservando o voxel da junção. Quando habilitado, remove EP–EP isolados com menos de `min_isolated_length` voxels. Ramos JP–JP não são removidos por esses critérios.

Os limiares usam quantidade de voxels no caminho, não comprimento físico. Esta etapa é separada da retração das pontas feita na poda angular.

In [ ]:
esqueleto_filtrado = analysis.filter_skeleton(
    esqueleto_inicial,
    tam_min=config.min_branch_length,
    remove_isolated=config.remove_isolated_branches,
    isolated_min=config.min_isolated_length,
)
print(
    f"Antes: {int(esqueleto_inicial.sum())}; após filtro: {int(esqueleto_filtrado.sum())}"
)

## 7. Estimar direções 3D e podar por continuidade

Com `local_search=True`, `analysis.prune_skeleton` estima a direção de cada ramo por ajuste de linha SVD nos primeiros `local_range` voxels, orientando o vetor para fora da junção. Com `False`, usa a direção global até o outro nó.

Em cada junção, compara os pares de vetores, aceita o par de maior ângulo se atingir `config.max_angle` e repete entre os ramos ainda não pareados. Ramos sem parceiro são retraídos em até `retraction_dist` voxels a partir da junção.

**Limitação preservada:** o grafo é obtido uma vez no início da poda; não é recalculado a cada remoção. A seleção de pares é iterativa, mas a atualização da topologia não. Pares aceitos que compartilham uma junção ainda podem permanecer conectados e receber o mesmo rótulo posteriormente.

In [ ]:
esqueleto_podado = analysis.prune_skeleton(
    esqueleto_filtrado,
    local_search=config.local_search,
    local_range=config.local_range,
    min_angle=config.max_angle,
    retraction_dist=config.retraction_dist,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, esqueleto, titulo in zip(
    axes,
    (esqueleto_inicial, esqueleto_filtrado, esqueleto_podado),
    ("Inicial", "Após filtro de ramos", "Após poda angular"),
):
    ax.imshow(esqueleto[indice_z], cmap="gray")
    ax.set(title=titulo, xlabel="X", ylabel="Y")
fig.tight_layout()
plt.show()

## 8. Rotular componentes e expandir por watershed 3D

`label_fibers_watershed` recebe a máscara 3D e o esqueleto podado. Internamente rotula os componentes conexos do esqueleto, calcula a transformada de distância euclidiana e aplica watershed em sua versão negativa, limitado pela máscara.

Os marcadores são exibidos abaixo apenas para inspeção: a função os calcula internamente a partir do esqueleto booleano. Não há segmentação independente de planos nem etapa de propagação de IDs. Um componente ainda conectado em uma junção pode representar mais de uma fibra física nos cruzamentos da amostra.

In [ ]:
marcadores_3d = label(esqueleto_podado)
rotulos_3d = label_fibers_watershed(mascara, esqueleto_podado)
print(f"Instâncias detectadas: {np.count_nonzero(np.unique(rotulos_3d))}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, rotulos, titulo in zip(
    axes, (marcadores_3d, rotulos_3d), ("Marcadores 3D", "Watershed 3D")
):
    ax.imshow(label2rgb(rotulos[indice_z], bg_label=0))
    ax.set(title=titulo, xlabel="X", ylabel="Y")
fig.tight_layout()
plt.show()

## 9. Substituir a sequência explícita pela classe

Em uma aplicação, a chamada abaixo **substitui as etapas 4–8**: recebe o volume selecionado em tons de cinza e executa internamente recorte, Otsu, abertura, esqueleto, filtro, poda e watershed. Não passe uma máscara já pré-processada.

Somente neste roteiro processamos a entrada das duas formas, verificando igualdade exata dos arrays retornados. `raw_volume` preserva a entrada; `cropped_volume`, `binary_mask`, `pruned_skeleton` e `labels` compartilham o referencial após o recorte automático. A classe não devolve o deslocamento desse recorte para o TIFF original.

`verbose=True` mostra configuração, etapas numeradas, contagens e tempos. Use `verbose=False` para desativar o progresso, inclusive das funções internas, sem ocultar avisos ou exceções.

In [ ]:
pipeline = Direct3DFiberPipeline(config, verbose=True)
resultado = pipeline.segment(volume_selecionado)
for obtido, esperado in (
    (resultado.raw_volume, volume_selecionado),
    (resultado.cropped_volume, volume_recortado),
    (resultado.binary_mask, mascara),
    (resultado.pruned_skeleton, esqueleto_podado),
    (resultado.labels, rotulos_3d),
):
    np.testing.assert_array_equal(obtido, esperado)
print("Classe e sequência explícita: todos os arrays são iguais.")

## 10. Calcular comprimento e orientação das instâncias

Usamos o mesmo `measure_fibers` da metodologia 2D, agora sobre `resultado.labels`. Para cada rótulo, a função reesqueletiza a máscara em sua caixa delimitadora e usa o par de endpoints mais distante para o comprimento **euclidiano EP–EP** e o vetor de direção. Não mede o comprimento do caminho percorrido no esqueleto.

Com menos de dois endpoints, usa a extensão projetada no eixo principal SVD; degenerados mantêm comprimento zero e direção padrão X. `voxel_count` é a quantidade de voxels do esqueleto, **não** o volume da instância.

Os ângulos `acos(abs(componente))` estão entre 0° e 90°, sem distinguir sentidos opostos. O espaçamento `voxel_size=(dz, dy, dx)` afeta comprimento e direção. Com `(1, 1, 1)`, comprimentos ficam em unidades de voxel; substitua pelo espaçamento físico calibrado para obter unidades físicas.

Não aplicamos filtro IQR, limites arbitrários ou conversão fixa para micrômetros. Os histogramas abaixo usam todas as instâncias, com peso igual.

In [ ]:
voxel_size = (1.0, 1.0, 1.0)
registros = measure_fibers(resultado.labels, voxel_size=voxel_size)
medidas = pd.DataFrame(registros)
display(medidas.head(10))

if medidas.empty:
    print("Nenhuma instância medida; histogramas não gerados.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].hist(medidas["ep_ep_length"], bins=20, color="steelblue")
    axes[0].set(
        title="Comprimento EP–EP", xlabel="Unidade do espaçamento", ylabel="Instâncias"
    )
    for ax, eixo in zip(axes[1:], ("x", "y", "z")):
        ax.hist(
            medidas[f"angle_{eixo}"], bins=np.linspace(0, 90, 19), color="steelblue"
        )
        ax.set(
            title=f"Orientação em relação a {eixo.upper()}",
            xlabel="Ângulo (graus)",
            ylabel="Instâncias",
        )
    fig.tight_layout()
    plt.show()

## 11. Exportar opcionalmente

`salvar_resultados=False` mantém a execução sem escrita de arquivos. Se habilitado, salvamos os rótulos em TIFF e usamos `analyze_and_save` para recalcular as mesmas medidas e produzir CSV e histogramas padronizados. Estes últimos usam percentuais nas orientações, enquanto a figura anterior usa contagens.

Os arquivos de mesmo nome são substituídos em novas exportações. Sem instâncias, somente o CSV com cabeçalho é produzido, sem novos PNGs. O TIFF usa o referencial recortado, não as coordenadas globais do arquivo de entrada. O import de reporting seleciona Agg para exportação e por isso fica após as visualizações.

In [ ]:
salvar_resultados = False
pasta_saida = root / "results" / "notebook_examples" / "direct_3d"

if salvar_resultados:
    from fiber_analysis.reporting import analyze_and_save

    pasta_saida.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(
        pasta_saida / "labels.tif",
        resultado.labels,
        photometric="minisblack",
        metadata={"axes": "ZYX"},
    )
    relatorio = analyze_and_save(
        resultado.labels, pasta_saida, prefix="direct_3d", voxel_size=voxel_size
    )
    print(f"Resultados exportados para: {pasta_saida}")
else:
    print("Exportação desativada; nenhum arquivo foi salvo.")

## Interpretação e relação com a outra metodologia

O método evita a correspondência entre planos, mas não garante separar todas as fibras que se tocam. Pares de ramos aceitos podem compartilhar o mesmo componente, e uma instância detectada pode ser uma fibra incompleta ou uma fusão. Não interprete o número de rótulos como uma contagem validada de fibras físicas.

O código correspondente está em `src/utils/analysis.py` e `src/utils/segmentation.py`. [fiber_seg_2D.ipynb](fiber_seg_2D.ipynb) apresenta a segmentação planar seguida de reconstrução, usando o mesmo módulo de medidas para tornar explícitas suas definições comuns.

As métricas deste roteiro não substituem a PCA específica do experimento de Hermans em [hermans_slice_yz.ipynb](hermans_slice_yz.ipynb). Os exemplos documentam a implementação atual, inclusive suas convenções e limitações; não corrigem diferenças em relação ao manuscrito nem alteram algoritmos compartilhados.